### Overview

This example demonstrates different ways that the same code can be executed.

1. Define the coroutine function. 
    1. Cell 'magic' `%callers` prints a list of [Caller](https://fleming79.github.io/async-kernel/api/#async_kernel.caller.Caller) instances and the thread in which it is executing.
    2. A button is created and  and it runs a loop where the user s
2. Execute `demo` normally.
3. Execute `demo` concurrently in a task.
4. Execute `demo` in a task.

![Simple demo](https://github.com/user-attachments/assets/9a4935ba-6af8-4c9f-bc67-b256be368811)

In [4]:
import anyio
import ipywidgets as ipw

from async_kernel import Caller


async def demo():
    %callers
    caller = Caller()  # Use caller set the event in the waiting thread
    b = ipw.Button(description="Continue")
    display(b)
    for i in range(1, 4):
        b.description = f"Continue {i}"
        event = anyio.Event()
        b.on_click(lambda _: caller.call_soon(event.set))  # noqa: B023
        print(f"Waiting {i}", end="\r")
        await event.wait()
    b.close()
    print("\nDone!")

In [5]:
await demo()

Active	Protected			Name
──────────────────────────────────────────────────────────────────────
   ✓	   🔐		MainThread	← current thread
   ✓	   🔐		ControlThread	


Button(description='Continue', style=ButtonStyle())

Waiting 3
Done!


In [6]:
##task
await demo()

Active	Protected			Name
──────────────────────────────────────────────────────────────────────
   ✓	   🔐		MainThread	← current thread
   ✓	   🔐		ControlThread	


Button(description='Continue', style=ButtonStyle())

Waiting 3
Done!


In [7]:
##thread
await demo()

Active	Protected			Name
──────────────────────────────────────────────────────────────────────
   ✓	   🔐		MainThread	
   ✓	   🔐		ControlThread	
   ✓			Thread-3 (anyio_run_caller)	← current thread


Button(description='Continue', style=ButtonStyle())

Waiting 3
Done!


[test](callers.ipynb)

## Caller.as_completed

See also: the [caller](https://fleming79.github.io/async-kernel/caller/) notebook.

In [28]:
# 

async for _ in Caller.as_completed( Caller().call_soon(demo) for _ in range(2)):
    pass

Active	Protected			Name
──────────────────────────────────────────────────────────────────────
   ✓	   🔐		MainThread	← current thread
   ✓	   🔐		ControlThread	
   ✓			Thread-3 (anyio_run_caller)	


Button(description='Continue', style=ButtonStyle())

Active	Protected			Name
──────────────────────────────────────────────────────────────────────
   ✓	   🔐		MainThread	← current thread
   ✓	   🔐		ControlThread	
   ✓			Thread-3 (anyio_run_caller)	


Button(description='Continue', style=ButtonStyle())

Waiting 3
Done!

Done!
